In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from gsm_benchmarker.results_analyser import MultiModelResultsAnalyser
from gsm_benchmarker.benchmark.answer_extractor import AnswerExtractor


plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()
p_standard = pp / "mini_20x50x4__14_11/final"
p_sep = pp / 'mini_sep_new__20x50__20_12/final'
p_code = pp / 'mini_code_output_20x50__05_12/final'

In [ ]:
results_standard = MultiModelResultsAnalyser(p_standard/'main_test', load_full_data=True)
results_sep = MultiModelResultsAnalyser(p_sep/'main_test', load_full_data=True)
results_code = MultiModelResultsAnalyser(p_code/'main_test', load_full_data=True)

## Babblers vs new prompt format

In [ ]:
results_code.get_babbler_counts().family

In [ ]:
fig = results_standard.plot_babblers_by_family()
_ = fig.suptitle("Babbler counts on GSM-Symbolic prompt")

In [ ]:
fig = results_sep.plot_babblers_by_family()
_ = fig.suptitle("Babbler counts on the anti-babbling prompt")

In [ ]:
results_standard.compare_babblers(results_sep, "Standard (GSM-Symbolic)", "Separated targets (anti-babbling)")


In [ ]:
results_sep.full_data

In [ ]:
sep_prep = results_sep.full_data.set_index(['model', 'id', 'instance']).drop(['original_id', 'question', 'answer', 'numerical_result'], axis=1)
standard_prep = results_standard.full_data.set_index(['model', 'id', 'instance'])

standard_prep.columns = [('standard_' + c) if c in sep_prep.columns else c for c in standard_prep.columns]
sep_prep.columns = ['sep_' + c for c in sep_prep.columns]

combined = pd.concat([standard_prep, sep_prep], axis=1)
combined = combined.reset_index()
# combined

### Effect on accuracy

In [ ]:
acc_down = combined[combined.standard_correct & ~combined.sep_correct]
acc_down

In [ ]:
acc_up = combined[~combined.standard_correct & combined.sep_correct]
acc_up

In [ ]:
def print_ex(qq):
    print("QUESTION")
    print(qq["question"])
    print(f"\nEXPECTED RESULT: {qq['numerical_result']}")
    print("\nSTANDARD PROMPT")
    print(AnswerExtractor.trim_response(qq["standard_full_response"]))
    print("\nSEPARATED TARGET")
    print(AnswerExtractor.trim_response(qq["sep_full_response"]))

In [ ]:
print_ex(acc_down.iloc[0])

In [ ]:
print_ex(acc_down.iloc[8])

In [ ]:
print_ex(acc_up.iloc[0])

---

# Code output prompt

In [ ]:
results_code.summary_data

In [ ]:
comp = pd.concat([results_standard.summary_data.accuracy, results_sep.summary_data.accuracy, results_code.summary_data.accuracy], axis=1) * 100
comp.columns = ['accuracy - standard', 'accuracy - separated_targets', 'accuracy - code']


In [ ]:
failed = results_code.full_data[results_code.full_data.predicted_numerical_result.isna()]

reasons = []
keep = []
for idx in range(len(failed)):
    row = failed.iloc[idx]
    res_, err = AnswerExtractor.extract_answer_code(row['full_response'])
    if res_ is None:
        reasons.append(err.name)
        keep.append(True)
    else:
        reasons.append("")
        keep.append(False)
failed["reason"] = reasons
failed = failed[keep]
failed.head(5)

In [ ]:
perc_failed  = failed.model.value_counts() / results_code.full_data.model.value_counts() * 100
perc_failed.name = "% failed answers"
perc_failed

In [ ]:
failed.reason.value_counts()

In [ ]:
def print_qq(qq):
    print(f"MODEL: {qq['model']}")
    print("\nQUESTION")
    print(qq["question"])
    print(f"\nEXPECTED RESULT: {qq['numerical_result']}")
    print("\nANSWER")
    print(10*'-')
    t = AnswerExtractor.trim_response(qq["full_response"])
    print(t)
    if len(t) != len(qq['full_response']):
        print("...")
    print(10*'-')

    ret, e = AnswerExtractor.extract_answer_code(qq['full_response'])
    if ret is None:
        print("\nISSUE")
        print(e.name)

In [ ]:
print_qq(failed.iloc[0])

In [ ]:
print_qq(failed.iloc[1])

In [ ]:
print_qq(failed.iloc[22])

In [ ]:
print_qq(failed.iloc[24])

In [ ]:
print_qq(failed.iloc[41])